In [ ]:
# ── COLAB SETUP ───────────────────────────────────────────────────────────────
# Run this cell first on Colab to install deps, clone the repo, and fetch the
# birefnet-preprocessed dataset. No-op when running locally.
import os, sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    REPO_URL = 'https://github.com/AlessandroGhiotto/industrial-AD2L.git'
    REPO_DIR = '/content/industrial-AD2L'
    if not os.path.exists(REPO_DIR):
        !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip install -q timm gdown
    DATA_DIR = 'dataset/adl-2025-2026-anomaly-detection_birefnet'
    if not os.path.exists(DATA_DIR):
        !gdown -q 'https://drive.google.com/uc?id=1lCqjjtKOhqoU3M5-4ktgiVlWnp6IFDYC' -O /content/dataset.zip
        !mkdir -p dataset && unzip -q /content/dataset.zip -d dataset/ && rm /content/dataset.zip
    print('Colab ready — CWD:', os.getcwd())


In [ ]:
# ── LOCAL SETUP ───────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings('ignore', message='No positive class found in y_true')
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent)) # Add repo root to path
sys.path.append(str(Path.cwd()))               # Add current 'final' folder to path

from src import config_final as cfg
from src import patchcore_final as pc
cfg.ensure_dirs()

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"Dataset Directory: {cfg.DATASET_DIR}")
print(f"Output Directory: {cfg.OUTPUT_DIR}")

# PatchCore with DINOv2
Implementation using:
- **DINOv2** backbone (ViT-B/14)
- **GPU-based PCA** for dimensionality reduction
- **Coreset selection** for memory efficiency
- **k-NN** anomaly scoring

## Architecture & Feature Extraction

In [ ]:
import torchvision.transforms as T
from PIL import Image
import numpy as np
import torch

extractor = pc.DINOv2(
    backbone_name=cfg.PC_BACKBONE, 
    layers=cfg.PC_LAYERS, 
    grid_size=cfg.PC_GRID_SIZE, 
    device=DEVICE
).to(DEVICE).eval()

_tfm = T.Compose([
    T.Resize((cfg.PC_IMAGE_SIZE, cfg.PC_IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def load_img(path):
    return _tfm(Image.open(path).convert('RGB'))

## Build or load normal-image feature banks

In [ ]:
import json
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
import numpy as np
import torch

class _DS(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return load_img(self.paths[i])

LOAD_BANKS = False
FEAT_DIM = len(cfg.PC_LAYERS) * extractor.backbone.embed_dim
_layers_str = '_'.join(map(str, cfg.PC_LAYERS))
_bank_dir = cfg.PC_BANKS_DIR / f'L{_layers_str}_pca{cfg.PC_PCA_DIM}'
classes = sorted([d.name for d in cfg.DATASET_DIR.iterdir() if d.is_dir() and d.name.startswith('class_')])

if LOAD_BANKS:
    print(f"Loading banks from {_bank_dir}...")
    pca = pc.GPUPCA(n_components=cfg.PC_PCA_DIM, device=DEVICE)
    pca.mean_ = torch.from_numpy(np.load(_bank_dir / 'pca_mean.npy')).to(DEVICE)
    pca.components_ = torch.from_numpy(np.load(_bank_dir / 'pca_components.npy')).to(DEVICE)
    banks = {}
    for cls in classes:
        arr = np.load(_bank_dir / f'{cls}_bank.npy')
        banks[cls] = torch.from_numpy(arr).to(DEVICE)
else:
    # ── Pass 1: collect a small per-class subsample, fit PCA, then discard ────
    # Peak CPU here ~= n_pca_per_class * n_classes * FEAT_DIM * 4 bytes (~1.6 GB)
    n_pca_per_class = 100_000 // len(classes)
    pca_pool = []
    for cls in classes:
        good_dir = cfg.DATASET_DIR / cls / 'train' / 'good'
        paths = sorted(good_dir.glob('*.png'))
        n_batches = max(1, (len(paths) + cfg.PC_BATCH_SIZE - 1) // cfg.PC_BATCH_SIZE)
        keep_pb = max(1, n_pca_per_class // n_batches)
        loader = DataLoader(_DS(paths), batch_size=cfg.PC_BATCH_SIZE, num_workers=4)
        cls_chunks = []
        with torch.no_grad(), torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            for batch in tqdm(loader, desc=f'PCA-fit {cls}', leave=False):
                p = extractor(batch.to(DEVICE)).float().cpu().numpy().reshape(-1, FEAT_DIM)
                if len(p) > keep_pb:
                    p = p[np.random.choice(len(p), keep_pb, replace=False)]
                cls_chunks.append(p)
        arr = np.concatenate(cls_chunks); del cls_chunks
        if len(arr) > n_pca_per_class:
            arr = arr[np.random.choice(len(arr), n_pca_per_class, replace=False)]
        pca_pool.append(arr)
    pca_fit_data = np.concatenate(pca_pool); del pca_pool
    print(f"Fitting PCA on {len(pca_fit_data)} samples (dim {FEAT_DIM} -> {cfg.PC_PCA_DIM})...")
    pca = pc.GPUPCA(n_components=cfg.PC_PCA_DIM, device=DEVICE).fit(pca_fit_data)
    del pca_fit_data

    # ── Pass 2: re-extract per class, transform on-the-fly, store post-PCA bank
    # Per-class CPU peak ~= PC_CORESET_SIZE * PC_PCA_DIM * 4 bytes (~200 MB at 100k/512)
    banks = {}
    for cls in classes:
        good_dir = cfg.DATASET_DIR / cls / 'train' / 'good'
        paths = sorted(good_dir.glob('*.png'))
        n_batches = max(1, (len(paths) + cfg.PC_BATCH_SIZE - 1) // cfg.PC_BATCH_SIZE)
        keep_pb = max(1, cfg.PC_CORESET_SIZE // n_batches)
        loader = DataLoader(_DS(paths), batch_size=cfg.PC_BATCH_SIZE, num_workers=4)
        pca_chunks = []
        with torch.no_grad(), torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            for batch in tqdm(loader, desc=f'bank {cls}', leave=False):
                p = extractor(batch.to(DEVICE)).float().cpu().numpy().reshape(-1, FEAT_DIM)
                if len(p) > keep_pb:
                    p = p[np.random.choice(len(p), keep_pb, replace=False)]
                pca_chunks.append(pca.transform(p).astype(np.float32))
        patches = np.concatenate(pca_chunks); del pca_chunks
        if len(patches) > cfg.PC_CORESET_SIZE:
            patches = patches[np.random.choice(len(patches), cfg.PC_CORESET_SIZE, replace=False)]
        banks[cls] = torch.from_numpy(patches).to(DEVICE)

    _bank_dir.mkdir(parents=True, exist_ok=True)
    np.save(_bank_dir / 'pca_mean.npy', pca.mean_.cpu().numpy())
    np.save(_bank_dir / 'pca_components.npy', pca.components_.cpu().numpy())
    for cls, b in banks.items(): np.save(_bank_dir / f'{cls}_bank.npy', b.cpu().numpy())

## Scoring & Calibration

In [ ]:
import random
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
import numpy as np
import torch

print('Computing training-normal calibration ...')
train_calib = {}
for cls in classes:
    all_paths = sorted((cfg.DATASET_DIR / cls / 'train' / 'good').glob('*.png'))
    paths = random.sample(all_paths, min(cfg.PC_CALIB_N, len(all_paths)))
    bank = banks[cls]
    max_vals = []
    loader = DataLoader(_DS(paths), batch_size=cfg.PC_BATCH_SIZE, num_workers=0)
    with torch.no_grad():
        for batch in tqdm(loader, desc=cls, leave=False):
            b_n = batch.shape[0]
            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                feats = extractor(batch.to(DEVICE)).float()
            fp = pca.transform(feats.cpu().numpy().reshape(-1, len(cfg.PC_LAYERS) * 1024))
            scores = pc.score_patches(fp, bank, device=DEVICE, knn_k=cfg.PC_KNN_K, score_chunk=cfg.PC_SCORE_CHUNK)
            hms = pc.make_heatmaps(scores.reshape(b_n, cfg.PC_GRID_SIZE**2), cfg.PC_GRID_SIZE, cfg.PC_SMOOTH_SIGMA, device=DEVICE)
            max_vals.extend(hms.reshape(b_n, -1).max(axis=1).tolist())
    train_calib[cls] = float(np.percentile(max_vals, cfg.PC_CALIB_HIGH))

raw_heatmaps = {}
for cls in classes:
    test_paths = sorted((cfg.DATASET_DIR / cls / 'test').glob('*.png'))
    bank = banks[cls]
    loader = DataLoader(_DS(test_paths), batch_size=cfg.PC_BATCH_SIZE, num_workers=0)
    with torch.no_grad():
        for bi, batch in enumerate(tqdm(loader, desc=cls, leave=False)):
            b_paths = test_paths[bi * cfg.PC_BATCH_SIZE : (bi + 1) * cfg.PC_BATCH_SIZE]
            b_n = len(b_paths)
            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                feats = extractor(batch.to(DEVICE)).float()
            fp = pca.transform(feats.cpu().numpy().reshape(-1, len(cfg.PC_LAYERS) * 1024))
            scores = pc.score_patches(fp, bank, device=DEVICE, knn_k=cfg.PC_KNN_K, score_chunk=cfg.PC_SCORE_CHUNK)
            hms = pc.make_heatmaps(scores.reshape(b_n, cfg.PC_GRID_SIZE**2), cfg.PC_GRID_SIZE, cfg.PC_SMOOTH_SIGMA, device=DEVICE)
            for j, p in enumerate(b_paths): raw_heatmaps[str(p)] = hms[j]

calibrated = {}
for p_str, hm in raw_heatmaps.items():
    cls = Path(p_str).parent.parent.name
    hi = train_calib[cls]
    calibrated[p_str] = np.clip(hm / (hi + 1e-8), 0., 1.)

In [ ]:
from src import visualize_final as viz
from sklearn.metrics import average_precision_score
import torch
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

N_VIS = 10
viz_outputs = []

for cls in classes:
    gt_root = cfg.DATASET_DIR / cls / 'ground_truth_train'
    trn_root = cfg.DATASET_DIR / cls / 'train'
    bank = banks[cls]
    
    examples = []
    for anom_dir in sorted(trn_root.iterdir()):
        if not anom_dir.name.startswith('anomaly_'): continue
        gt_dir = gt_root / anom_dir.name
        for p in sorted(anom_dir.glob('*.png')):
            mp = gt_dir / p.name
            if mp.exists():
                examples.append((p, mp, anom_dir.name))
                break
        if len(examples) >= N_VIS: break
    
    if not examples: continue
    
    for img_path, mask_path, anom_name in tqdm(examples, desc=f"Viz {cls}"):
        img_pil = Image.open(img_path).convert('RGB')
        p_in = pc.load_img(img_path).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feats = extractor(p_in).cpu().numpy().reshape(-1, len(cfg.PC_LAYERS) * 1024)
        fp = pca.transform(feats)
        scores = pc.score_patches(fp, bank, device=DEVICE)
        hm = pc.make_heatmaps(scores.reshape(1, cfg.PC_GRID_SIZE**2), cfg.PC_GRID_SIZE, cfg.PC_SMOOTH_SIGMA, device=DEVICE, out_size=cfg.PC_IMAGE_SIZE)[0]
        
        mask = viz.load_mask(str(mask_path), size=(cfg.PC_IMAGE_SIZE, cfg.PC_IMAGE_SIZE))
        if mask.shape != hm.shape:
            mask = np.array(Image.fromarray(mask).resize((hm.shape[1], hm.shape[0]), Image.NEAREST))
        ap = average_precision_score(mask.ravel().astype(int), hm.ravel())
        
        viz_outputs.append({
            'input': img_pil,
            'gt': mask,
            'heatmap': hm,
            'class_name': cls,
            'anomaly_name': anom_name,
            'ap': ap
        })

pdf_path = cfg.OUTPUT_DIR / 'patchcore_visualizations.pdf'
viz.export_to_pdf(viz_outputs, str(pdf_path), title_prefix="PatchCore Results:")

## Generate submission CSV

In [ ]:
import pandas as pd
from pathlib import Path

rows = []
for p_str, hm in calibrated.items():
    rows.append({'ID': Path(p_str).stem, 'Label': pc.rle_encode_q8(hm)})
sub = pd.DataFrame(rows).sort_values('ID').reset_index(drop=True)
sub_path = cfg.PC_SUB_DIR / 'submission_patchcore_final.csv'
sub.to_csv(sub_path, index=False)
print(f"Submission saved to {sub_path}")